### Pass@k Explanation

Pass@k is a metric used to evaluate the performance of code generation models. Unlike Pass@1, which measures if the first generated solution is correct, Pass@k checks if at least one of `k` independently generated solutions for a given problem is correct.

Here's how it works:

1.  For each problem, the model generates `k` different code solutions.
2.  Each of these `k` solutions is tested against the problem's test cases.
3.  If *any* of the `k` solutions pass all the tests, the problem is considered 'solved'.
4.  Pass@k is then calculated as the proportion of problems that are solved (i.e., for which at least one of the `k` solutions passed) out of the total number of problems.

This metric provides a more robust evaluation of a model's capabilities, especially when the model might not consistently generate the best solution as its top choice but can generate several plausible ones.

Pass@1 for text to python

| Metric             | Value      |
| ------------------ | ---------- |
| Total Solved       | **429**    |
| Total Problems     | **974**    |
| Overall Pass@1     | **0.4405** |
| Overall Pass@1 (%) | **44.05%** |

max_new_tokens=300, do_sample=True,  temperature=0.2, top_p=0.95

Pass@1 for text to python

| Metric             | Value      |
| ------------------ | ---------- |
| Total Solved       | **385**    |
| Total Problems     | **974**    |
| Overall Pass@1     | **0.3953** |
| Overall Pass@1 (%) | **39.53%** |

max_new_tokens=300,
do_sample=True,
temperature=0.7,


Pass@10 for text to python

| Metric              | Value      |
| ------------------- | ---------- |
| Total Solved        | **71**     |
| Total Problems      | **100**    |
| Overall Pass@10     | **0.7100** |
| Overall Pass@10 (%) | **71.00%** |

max_new_tokens=300, do_sample=True, temperature=0.7,

In [1]:
!pip install -q datasets==3.6.0
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 20.5 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, PeftModel
from google.colab import userdata
import datasets
from datasets import load_dataset, Dataset
import torch
import pandas as pd

In [3]:
print(datasets.__version__)

3.6.0


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

HF_TOKEN = userdata.get('HF_TOKEN')
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "bigcode/starcoder2-3b"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, token=HF_TOKEN, quantization_config=bnb_config).to(device)

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [5]:
!unzip -o /content/starcoder2-python-java-custom-lora.zip -d /content/starcoder2-python-java-custom-lora/

Archive:  /content/starcoder2-python-java-custom-lora.zip
  inflating: /content/starcoder2-python-java-custom-lora/adapter_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer.json  
  inflating: /content/starcoder2-python-java-custom-lora/adapter_model.safetensors  
  inflating: /content/starcoder2-python-java-custom-lora/training_args.bin  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/README.md  


In [6]:
# Load the LoRA adapters from the checkpoint
model_to_test = PeftModel.from_pretrained(model, "/content/starcoder2-python-java-custom-lora")

# Set the model to evaluation mode and move to device
model_to_test = model_to_test.eval().to(device)

print("Model loaded successfully for testing.")

Model loaded successfully for testing.


In [7]:
def generate_solution(prompt, test_list, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             {prompt}
             ### Test Cases
             {test_list}
             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    # #Greedy decoding
    # outputs = model_to_test.generate(
    # **inputs,
    # max_new_tokens=300,
    # do_sample=False,
    #repetition_penalty=1.2,
    #num_return_sequences=num_solutions,
    # eos_token_id=tokenizer.eos_token_id,
    # pad_token_id=tokenizer.eos_token_id

    # )

    #Low temperature
    outputs = model_to_test.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        #repetition_penalty=1.2,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    # for pass@10, 20
    # outputs = model_to_test.generate(
    #     **inputs,
    #     max_new_tokens=300,
    #     do_sample=True, # Enable sampling for diverse solutions
    #     temperature=0.7, # Add temperature for diversity
    #     num_return_sequences=num_solutions, # Generate multiple solutions
    #     eos_token_id=tokenizer.eos_token_id
    # )

    # Decode all generated sequences
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

In [8]:
def extract_code(text):
    if "### Response" in text:
        return text.split("### Response")[-1].strip()

    return text.strip()

def run_mbpp_tests(code, test_list):
    namespace = {}

    try:
        exec(code, namespace)

        for test in test_list:
            exec(test, namespace)

        return True

    except Exception:
        return False

In [9]:
def evaluate_pass_at_k(ds, k_value):
    solved_problems = 0
    total_problems = len(ds)
    print(f"\n--- Evaluating Pass@{k_value} ---")

    for i, row in enumerate(ds, 1):
        # Generate k solutions for each problem
        generated_solutions = generate_solution(row["text"], row["test_list"], num_solutions=k_value)

        problem_solved = False
        for solution_text in generated_solutions:
            code = extract_code(solution_text)
            success = run_mbpp_tests(
                code,
                row["test_list"]
            )
            if success:
                problem_solved = True
                break # Found a passing solution, no need to check others for this problem

        if problem_solved:
            solved_problems += 1

        print(f"Problem {i}/{total_problems}: {'SOLVED' if problem_solved else 'FAILED'}. Total solved: {solved_problems}/{total_problems}")

    pass_at_k = solved_problems / total_problems
    print(f"\nPass@{k_value} = {pass_at_k:.4f}")
    return pass_at_k

In [10]:
mbpp = load_dataset("Muennighoff/mbpp", trust_remote_code=True)
test_ds = mbpp['test']

README.md:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

mbpp.py:   0%|          | 0.00/3.59k [00:00<?, ?B/s]

mbpp.jsonl:   0%|          | 0.00/564k [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
# Use .select() to get a slice as a new Dataset object, then .to_list() to convert it to a list of dictionaries.
#pass_at_10 = evaluate_pass_at_k(test_ds.select(range(0,10)).to_list(), k_value=1)
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(0,50)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: FAILED. Total solved: 0/50
Problem 4/50: FAILED. Total solved: 0/50
Problem 5/50: SOLVED. Total solved: 1/50
Problem 6/50: FAILED. Total solved: 1/50
Problem 7/50: SOLVED. Total solved: 2/50
Problem 8/50: SOLVED. Total solved: 3/50


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Problem 9/50: FAILED. Total solved: 3/50
Problem 10/50: SOLVED. Total solved: 4/50


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Problem 11/50: SOLVED. Total solved: 5/50
Problem 12/50: FAILED. Total solved: 5/50


...

Problem 13/50: FAILED. Total solved: 5/50
Problem 14/50: FAILED. Total solved: 5/50
Problem 15/50: FAILED. Total solved: 5/50


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Problem 16/50: SOLVED. Total solved: 6/50
Problem 17/50: SOLVED. Total solved: 7/50


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Problem 18/50: SOLVED. Total solved: 8/50
Problem 19/50: SOLVED. Total solved: 9/50
Problem 20/50: FAILED. Total solved: 9/50
Problem 21/50: FAILED. Total solved: 9/50
Problem 22/50: SOLVED. Total solved: 10/50
Problem 23/50: SOLVED. Total solved: 11/50
Problem 24/50: FAILED. Total solved: 11/50
Problem 25/50: FAILED. Total solved: 11/50
Problem 26/50: FAILED. Total solved: 11/50
Problem 27/50: FAILED. Total solved: 11/50
Problem 28/50: SOLVED. Total solved: 12/50
Problem 29/50: SOLVED. Total solved: 13/50
Problem 30/50: FAILED. Total solved: 13/50
Problem 31/50: FAILED. Total solved: 13/50
Problem 32/50: SOLVED. Total solved: 14/50
Problem 33/50: FAILED. Total solved: 14/50
Problem 34/50: SOLVED. Total solved: 15/50
Problem 35/50: FAILED. Total solved: 15/50
Problem 36/50: FAILED. Total solved: 15/50
Problem 37/50: FAILED. Total solved: 15/50
Problem 38/50: FAILED. Total solved: 15/50
Problem 39/50: FAILED. Total solved: 15/50
Problem 40/50: SOLVED. Total solved: 16/50
Problem 41/50: 

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(50,100)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: SOLVED. Total solved: 1/50
Problem 2/50: SOLVED. Total solved: 2/50
Problem 3/50: SOLVED. Total solved: 3/50
Problem 4/50: SOLVED. Total solved: 4/50
Problem 5/50: FAILED. Total solved: 4/50
Problem 6/50: FAILED. Total solved: 4/50
Problem 7/50: FAILED. Total solved: 4/50
Problem 8/50: SOLVED. Total solved: 5/50
Problem 9/50: FAILED. Total solved: 5/50
Problem 10/50: FAILED. Total solved: 5/50
Problem 11/50: SOLVED. Total solved: 6/50
Problem 12/50: SOLVED. Total solved: 7/50
Problem 13/50: FAILED. Total solved: 7/50
Problem 14/50: SOLVED. Total solved: 8/50
Problem 15/50: SOLVED. Total solved: 9/50
Problem 16/50: SOLVED. Total solved: 10/50
Problem 17/50: FAILED. Total solved: 10/50
Problem 18/50: FAILED. Total solved: 10/50
Problem 19/50: SOLVED. Total solved: 11/50
Problem 20/50: FAILED. Total solved: 11/50
Problem 21/50: FAILED. Total solved: 11/50
Problem 22/50: FAILED. Total solved: 11/50
Problem 23/50: FAILED. Total solved: 11/50
Problem 

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(100,150)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: FAILED. Total solved: 1/50
Problem 4/50: FAILED. Total solved: 1/50
Problem 5/50: SOLVED. Total solved: 2/50
Problem 6/50: FAILED. Total solved: 2/50
Problem 7/50: SOLVED. Total solved: 3/50
Problem 8/50: SOLVED. Total solved: 4/50
Problem 9/50: SOLVED. Total solved: 5/50
Problem 10/50: FAILED. Total solved: 5/50
Problem 11/50: FAILED. Total solved: 5/50
Problem 12/50: SOLVED. Total solved: 6/50
Problem 13/50: SOLVED. Total solved: 7/50
Problem 14/50: FAILED. Total solved: 7/50
Problem 15/50: SOLVED. Total solved: 8/50
Problem 16/50: SOLVED. Total solved: 9/50
Problem 17/50: FAILED. Total solved: 9/50
Problem 18/50: SOLVED. Total solved: 10/50
Problem 19/50: FAILED. Total solved: 10/50
Problem 20/50: FAILED. Total solved: 10/50
Problem 21/50: FAILED. Total solved: 10/50
Problem 22/50: SOLVED. Total solved: 11/50
Problem 23/50: FAILED. Total solved: 11/50
Problem 24

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(150,200)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: SOLVED. Total solved: 1/50
Problem 2/50: SOLVED. Total solved: 2/50
Problem 3/50: SOLVED. Total solved: 3/50
Problem 4/50: SOLVED. Total solved: 4/50
Problem 5/50: FAILED. Total solved: 4/50
Problem 6/50: FAILED. Total solved: 4/50
Problem 7/50: SOLVED. Total solved: 5/50
Problem 8/50: FAILED. Total solved: 5/50
Problem 9/50: FAILED. Total solved: 5/50
Problem 10/50: FAILED. Total solved: 5/50
Problem 11/50: SOLVED. Total solved: 6/50
Problem 12/50: FAILED. Total solved: 6/50
Problem 13/50: FAILED. Total solved: 6/50
Problem 14/50: FAILED. Total solved: 6/50
Problem 15/50: FAILED. Total solved: 6/50
Problem 16/50: SOLVED. Total solved: 7/50
Problem 17/50: FAILED. Total solved: 7/50
Problem 18/50: SOLVED. Total solved: 8/50
Problem 19/50: SOLVED. Total solved: 9/50
Problem 20/50: SOLVED. Total solved: 10/50
Problem 21/50: SOLVED. Total solved: 11/50
Problem 22/50: FAILED. Total solved: 11/50
Problem 23/50: SOLVED. Total solved: 12/50
Problem 24/5

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(200,250)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: SOLVED. Total solved: 1/50
Problem 2/50: FAILED. Total solved: 1/50
Problem 3/50: SOLVED. Total solved: 2/50
Problem 4/50: SOLVED. Total solved: 3/50
Problem 5/50: FAILED. Total solved: 3/50
Problem 6/50: FAILED. Total solved: 3/50
Problem 7/50: FAILED. Total solved: 3/50


<string>:2: SyntaxWarning: invalid escape sequence '\.'


Problem 8/50: FAILED. Total solved: 3/50
Problem 9/50: FAILED. Total solved: 3/50
Problem 10/50: FAILED. Total solved: 3/50
Problem 11/50: FAILED. Total solved: 3/50
Problem 12/50: SOLVED. Total solved: 4/50
Problem 13/50: FAILED. Total solved: 4/50
Problem 14/50: FAILED. Total solved: 4/50
Problem 15/50: FAILED. Total solved: 4/50
Problem 16/50: FAILED. Total solved: 4/50
Problem 17/50: FAILED. Total solved: 4/50
Problem 18/50: FAILED. Total solved: 4/50
Problem 19/50: FAILED. Total solved: 4/50
Problem 20/50: FAILED. Total solved: 4/50
Problem 21/50: SOLVED. Total solved: 5/50
Problem 22/50: SOLVED. Total solved: 6/50
Problem 23/50: SOLVED. Total solved: 7/50
Problem 24/50: SOLVED. Total solved: 8/50
Problem 25/50: FAILED. Total solved: 8/50
Problem 26/50: SOLVED. Total solved: 9/50
Problem 27/50: SOLVED. Total solved: 10/50
Problem 28/50: FAILED. Total solved: 10/50
Problem 29/50: FAILED. Total solved: 10/50
Problem 30/50: SOLVED. Total solved: 11/50
Problem 31/50: FAILED. Total sol

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(250,300)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: SOLVED. Total solved: 1/50
Problem 4/50: FAILED. Total solved: 1/50
Problem 5/50: FAILED. Total solved: 1/50
Problem 6/50: FAILED. Total solved: 1/50
Problem 7/50: SOLVED. Total solved: 2/50
Problem 8/50: SOLVED. Total solved: 3/50
Problem 9/50: FAILED. Total solved: 3/50
Problem 10/50: SOLVED. Total solved: 4/50
Problem 11/50: FAILED. Total solved: 4/50
Problem 12/50: SOLVED. Total solved: 5/50
Problem 13/50: SOLVED. Total solved: 6/50
Problem 14/50: FAILED. Total solved: 6/50
Problem 15/50: FAILED. Total solved: 6/50
Problem 16/50: FAILED. Total solved: 6/50
Problem 17/50: FAILED. Total solved: 6/50
Problem 18/50: FAILED. Total solved: 6/50
Problem 19/50: SOLVED. Total solved: 7/50
Problem 20/50: FAILED. Total solved: 7/50
Problem 21/50: FAILED. Total solved: 7/50
Problem 22/50: FAILED. Total solved: 7/50
Problem 23/50: SOLVED. Total solved: 8/50
Problem 24/50: S

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(300,350)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: FAILED. Total solved: 0/50
Problem 4/50: FAILED. Total solved: 0/50
Problem 5/50: FAILED. Total solved: 0/50
Problem 6/50: FAILED. Total solved: 0/50
Problem 7/50: FAILED. Total solved: 0/50
Problem 8/50: FAILED. Total solved: 0/50
Problem 9/50: SOLVED. Total solved: 1/50
Problem 10/50: FAILED. Total solved: 1/50
Problem 11/50: FAILED. Total solved: 1/50
Problem 12/50: FAILED. Total solved: 1/50
Problem 13/50: FAILED. Total solved: 1/50
Problem 14/50: FAILED. Total solved: 1/50
Problem 15/50: FAILED. Total solved: 1/50
Problem 16/50: FAILED. Total solved: 1/50
Problem 17/50: FAILED. Total solved: 1/50
Problem 18/50: FAILED. Total solved: 1/50
Problem 19/50: FAILED. Total solved: 1/50
Problem 20/50: FAILED. Total solved: 1/50
Problem 21/50: FAILED. Total solved: 1/50
Problem 22/50: FAILED. Total solved: 1/50
Problem 23/50: FAILED. Total solved: 1/50
Problem 24/50: F

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(350,400)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: FAILED. Total solved: 1/50
Problem 4/50: FAILED. Total solved: 1/50
Problem 5/50: FAILED. Total solved: 1/50
Problem 6/50: SOLVED. Total solved: 2/50
Problem 7/50: SOLVED. Total solved: 3/50
Problem 8/50: SOLVED. Total solved: 4/50
Problem 9/50: FAILED. Total solved: 4/50
Problem 10/50: SOLVED. Total solved: 5/50
Problem 11/50: SOLVED. Total solved: 6/50
Problem 12/50: SOLVED. Total solved: 7/50
Problem 13/50: SOLVED. Total solved: 8/50
Problem 14/50: FAILED. Total solved: 8/50
Problem 15/50: SOLVED. Total solved: 9/50
Problem 16/50: SOLVED. Total solved: 10/50
Problem 17/50: FAILED. Total solved: 10/50
Problem 18/50: FAILED. Total solved: 10/50
Problem 19/50: FAILED. Total solved: 10/50
Problem 20/50: FAILED. Total solved: 10/50
Problem 21/50: SOLVED. Total solved: 11/50
Problem 22/50: FAILED. Total solved: 11/50
Problem 23/50: SOLVED. Total solved: 12/50
Problem 

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(400,450)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: FAILED. Total solved: 1/50
Problem 4/50: SOLVED. Total solved: 2/50
Problem 5/50: FAILED. Total solved: 2/50
Problem 6/50: SOLVED. Total solved: 3/50
Problem 7/50: FAILED. Total solved: 3/50
Problem 8/50: FAILED. Total solved: 3/50
Problem 9/50: FAILED. Total solved: 3/50
Problem 10/50: FAILED. Total solved: 3/50
Problem 11/50: FAILED. Total solved: 3/50
Problem 12/50: SOLVED. Total solved: 4/50
Problem 13/50: SOLVED. Total solved: 5/50
Problem 14/50: SOLVED. Total solved: 6/50
Problem 15/50: FAILED. Total solved: 6/50
Problem 16/50: FAILED. Total solved: 6/50
Problem 17/50: FAILED. Total solved: 6/50
Problem 18/50: SOLVED. Total solved: 7/50
Problem 19/50: FAILED. Total solved: 7/50
Problem 20/50: FAILED. Total solved: 7/50
Problem 21/50: FAILED. Total solved: 7/50
Problem 22/50: SOLVED. Total solved: 8/50
Problem 23/50: FAILED. Total solved: 8/50
Problem 24/50: S

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(450,500)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: FAILED. Total solved: 0/50
Problem 4/50: SOLVED. Total solved: 1/50
Problem 5/50: SOLVED. Total solved: 2/50
Problem 6/50: SOLVED. Total solved: 3/50
Problem 7/50: FAILED. Total solved: 3/50
Problem 8/50: SOLVED. Total solved: 4/50
Problem 9/50: FAILED. Total solved: 4/50
Problem 10/50: FAILED. Total solved: 4/50
Problem 11/50: FAILED. Total solved: 4/50
Problem 12/50: FAILED. Total solved: 4/50
Problem 13/50: FAILED. Total solved: 4/50
Problem 14/50: SOLVED. Total solved: 5/50
Problem 15/50: SOLVED. Total solved: 6/50
Problem 16/50: FAILED. Total solved: 6/50
Problem 17/50: FAILED. Total solved: 6/50
Problem 18/50: FAILED. Total solved: 6/50
Problem 19/50: FAILED. Total solved: 6/50
Problem 20/50: FAILED. Total solved: 6/50
Problem 21/50: SOLVED. Total solved: 7/50
Problem 22/50: SOLVED. Total solved: 8/50
Problem 23/50: FAILED. Total solved: 8/50
Problem 24/50: S

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(500,550)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: FAILED. Total solved: 1/50
Problem 4/50: SOLVED. Total solved: 2/50
Problem 5/50: SOLVED. Total solved: 3/50
Problem 6/50: SOLVED. Total solved: 4/50
Problem 7/50: SOLVED. Total solved: 5/50
Problem 8/50: FAILED. Total solved: 5/50
Problem 9/50: SOLVED. Total solved: 6/50
Problem 10/50: FAILED. Total solved: 6/50
Problem 11/50: SOLVED. Total solved: 7/50
Problem 12/50: FAILED. Total solved: 7/50
Problem 13/50: FAILED. Total solved: 7/50
Problem 14/50: SOLVED. Total solved: 8/50
Problem 15/50: FAILED. Total solved: 8/50
Problem 16/50: SOLVED. Total solved: 9/50
Problem 17/50: SOLVED. Total solved: 10/50
Problem 18/50: SOLVED. Total solved: 11/50
Problem 19/50: FAILED. Total solved: 11/50
Problem 20/50: FAILED. Total solved: 11/50
Problem 21/50: FAILED. Total solved: 11/50
Problem 22/50: FAILED. Total solved: 11/50
Problem 23/50: FAILED. Total solved: 11/50
Problem 2

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(550,600)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: SOLVED. Total solved: 1/50
Problem 2/50: FAILED. Total solved: 1/50
Problem 3/50: FAILED. Total solved: 1/50
Problem 4/50: FAILED. Total solved: 1/50
Problem 5/50: SOLVED. Total solved: 2/50
Problem 6/50: SOLVED. Total solved: 3/50
Problem 7/50: SOLVED. Total solved: 4/50
Problem 8/50: FAILED. Total solved: 4/50
Problem 9/50: SOLVED. Total solved: 5/50
Problem 10/50: FAILED. Total solved: 5/50
Problem 11/50: FAILED. Total solved: 5/50
Problem 12/50: SOLVED. Total solved: 6/50
Problem 13/50: FAILED. Total solved: 6/50
Problem 14/50: SOLVED. Total solved: 7/50
Problem 15/50: SOLVED. Total solved: 8/50
Problem 16/50: SOLVED. Total solved: 9/50
Problem 17/50: SOLVED. Total solved: 10/50
Problem 18/50: SOLVED. Total solved: 11/50
Problem 19/50: FAILED. Total solved: 11/50
Problem 20/50: FAILED. Total solved: 11/50
Problem 21/50: FAILED. Total solved: 11/50
Problem 22/50: FAILED. Total solved: 11/50
Problem 23/50: SOLVED. Total solved: 12/50
Problem 2

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(600,650)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: FAILED. Total solved: 0/50
Problem 4/50: SOLVED. Total solved: 1/50
Problem 5/50: SOLVED. Total solved: 2/50
Problem 6/50: SOLVED. Total solved: 3/50
Problem 7/50: SOLVED. Total solved: 4/50
Problem 8/50: FAILED. Total solved: 4/50
Problem 9/50: FAILED. Total solved: 4/50
Problem 10/50: FAILED. Total solved: 4/50
Problem 11/50: SOLVED. Total solved: 5/50
Problem 12/50: FAILED. Total solved: 5/50
Problem 13/50: FAILED. Total solved: 5/50
Problem 14/50: FAILED. Total solved: 5/50
Problem 15/50: FAILED. Total solved: 5/50
Problem 16/50: SOLVED. Total solved: 6/50
Problem 17/50: FAILED. Total solved: 6/50
Problem 18/50: SOLVED. Total solved: 7/50
Problem 19/50: FAILED. Total solved: 7/50
Problem 20/50: FAILED. Total solved: 7/50
Problem 21/50: SOLVED. Total solved: 8/50
Problem 22/50: FAILED. Total solved: 8/50
Problem 23/50: SOLVED. Total solved: 9/50
Problem 24/50: F

In [ ]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(650,700)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: SOLVED. Total solved: 1/50
Problem 4/50: SOLVED. Total solved: 2/50
Problem 5/50: SOLVED. Total solved: 3/50
Problem 6/50: SOLVED. Total solved: 4/50
Problem 7/50: SOLVED. Total solved: 5/50
Problem 8/50: SOLVED. Total solved: 6/50
Problem 9/50: FAILED. Total solved: 6/50
Problem 10/50: FAILED. Total solved: 6/50
Problem 11/50: FAILED. Total solved: 6/50
Problem 12/50: SOLVED. Total solved: 7/50
Problem 13/50: FAILED. Total solved: 7/50
Problem 14/50: SOLVED. Total solved: 8/50
Problem 15/50: SOLVED. Total solved: 9/50
Problem 16/50: SOLVED. Total solved: 10/50
Problem 17/50: FAILED. Total solved: 10/50
Problem 18/50: FAILED. Total solved: 10/50
Problem 19/50: FAILED. Total solved: 10/50
Problem 20/50: FAILED. Total solved: 10/50
Problem 21/50: FAILED. Total solved: 10/50
Problem 22/50: SOLVED. Total solved: 11/50
Problem 23/50: SOLVED. Total solved: 12/50
Problem 

In [13]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(699,750)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Problem 1/51: SOLVED. Total solved: 1/51
Problem 2/51: FAILED. Total solved: 1/51
Problem 3/51: FAILED. Total solved: 1/51
Problem 4/51: SOLVED. Total solved: 2/51
Problem 5/51: SOLVED. Total solved: 3/51
Problem 6/51: SOLVED. Total solved: 4/51
Problem 7/51: FAILED. Total solved: 4/51
Problem 8/51: FAILED. Total solved: 4/51
Problem 9/51: FAILED. Total solved: 4/51
Problem 10/51: SOLVED. Total solved: 5/51
Problem 11/51: SOLVED. Total solved: 6/51
Problem 12/51: FAILED. Total solved: 6/51
Problem 13/51: FAILED. Total solved: 6/51
Problem 14/51: SOLVED. Total solved: 7/51
Problem 15/51: FAILED. Total solved: 7/51
Problem 16/51: SOLVED. Total solved: 8/51
Problem 17/51: SOLVED. Total solved: 9/51
Problem 18/51: FAILED. Total solved: 9/51
Problem 19/51: SOLVED. Total solved: 10/51
Problem 20/51: FAILED. Total solved: 10/51
Problem 21/51: SOLVED. Total solved: 11/51
Problem 22/51: FAILED. Total solved: 11/51
Problem 23/51: FAILED. Total solved: 11/51
Problem 24/51: FAILED. Total solved: 1

In [14]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(750,800)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: SOLVED. Total solved: 2/50
Problem 4/50: FAILED. Total solved: 2/50
Problem 5/50: FAILED. Total solved: 2/50
Problem 6/50: FAILED. Total solved: 2/50
Problem 7/50: FAILED. Total solved: 2/50
Problem 8/50: FAILED. Total solved: 2/50
Problem 9/50: FAILED. Total solved: 2/50
Problem 10/50: FAILED. Total solved: 2/50
Problem 11/50: FAILED. Total solved: 2/50
Problem 12/50: SOLVED. Total solved: 3/50
Problem 13/50: FAILED. Total solved: 3/50
Problem 14/50: SOLVED. Total solved: 4/50
Problem 15/50: SOLVED. Total solved: 5/50
Problem 16/50: SOLVED. Total solved: 6/50
Problem 17/50: SOLVED. Total solved: 7/50
Problem 18/50: FAILED. Total solved: 7/50
Problem 19/50: FAILED. Total solved: 7/50
Problem 20/50: FAILED. Total solved: 7/50
Problem 21/50: FAILED. Total solved: 7/50
Problem 22/50: SOLVED. Total solved: 8/50
Problem 23/50: FAILED. Total solved: 8/50


<string>:3: SyntaxWarning: invalid escape sequence '\.'


Problem 24/50: SOLVED. Total solved: 9/50
Problem 25/50: SOLVED. Total solved: 10/50
Problem 26/50: FAILED. Total solved: 10/50
Problem 27/50: SOLVED. Total solved: 11/50
Problem 28/50: SOLVED. Total solved: 12/50
Problem 29/50: FAILED. Total solved: 12/50
Problem 30/50: SOLVED. Total solved: 13/50
Problem 31/50: SOLVED. Total solved: 14/50
Problem 32/50: FAILED. Total solved: 14/50
Problem 33/50: FAILED. Total solved: 14/50
Problem 34/50: SOLVED. Total solved: 15/50
Problem 35/50: SOLVED. Total solved: 16/50
Problem 36/50: SOLVED. Total solved: 17/50
Problem 37/50: FAILED. Total solved: 17/50
Problem 38/50: SOLVED. Total solved: 18/50
Problem 39/50: SOLVED. Total solved: 19/50
Problem 40/50: SOLVED. Total solved: 20/50
Problem 41/50: FAILED. Total solved: 20/50
Problem 42/50: SOLVED. Total solved: 21/50
Problem 43/50: FAILED. Total solved: 21/50
Problem 44/50: SOLVED. Total solved: 22/50
Problem 45/50: FAILED. Total solved: 22/50
Problem 46/50: SOLVED. Total solved: 23/50
Problem 47/5

In [15]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(800,850)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: FAILED. Total solved: 0/50
Problem 3/50: SOLVED. Total solved: 1/50
Problem 4/50: SOLVED. Total solved: 2/50
Problem 5/50: SOLVED. Total solved: 3/50
Problem 6/50: SOLVED. Total solved: 4/50
Problem 7/50: SOLVED. Total solved: 5/50
Problem 8/50: SOLVED. Total solved: 6/50
Problem 9/50: FAILED. Total solved: 6/50
Problem 10/50: SOLVED. Total solved: 7/50
Problem 11/50: SOLVED. Total solved: 8/50
Problem 12/50: SOLVED. Total solved: 9/50
Problem 13/50: SOLVED. Total solved: 10/50
Problem 14/50: SOLVED. Total solved: 11/50
Problem 15/50: FAILED. Total solved: 11/50
Problem 16/50: SOLVED. Total solved: 12/50
Problem 17/50: SOLVED. Total solved: 13/50
Problem 18/50: SOLVED. Total solved: 14/50
Problem 19/50: SOLVED. Total solved: 15/50
Problem 20/50: FAILED. Total solved: 15/50
Problem 21/50: SOLVED. Total solved: 16/50
Problem 22/50: FAILED. Total solved: 16/50
Problem 23/50: SOLVED. Total solved: 17/50
Probl

In [13]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(850,900)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: SOLVED. Total solved: 2/50
Problem 4/50: FAILED. Total solved: 2/50
Problem 5/50: SOLVED. Total solved: 3/50
Problem 6/50: FAILED. Total solved: 3/50
Problem 7/50: SOLVED. Total solved: 4/50
Problem 8/50: FAILED. Total solved: 4/50
Problem 9/50: FAILED. Total solved: 4/50
Problem 10/50: FAILED. Total solved: 4/50
Problem 11/50: SOLVED. Total solved: 5/50
Problem 12/50: SOLVED. Total solved: 6/50
Problem 13/50: FAILED. Total solved: 6/50
Problem 14/50: SOLVED. Total solved: 7/50
Problem 15/50: FAILED. Total solved: 7/50
Problem 16/50: SOLVED. Total solved: 8/50
Problem 17/50: FAILED. Total solved: 8/50
Problem 18/50: FAILED. Total solved: 8/50
Problem 19/50: FAILED. Total solved: 8/50
Problem 20/50: SOLVED. Total solved: 9/50
Problem 21/50: SOLVED. Total solved: 10/50
Problem 22/50: FAILED. Total solved: 10/50
Problem 23/50: SOLVED. Total solved: 11/50
Problem 24/50: FAILED. Total solved: 11/

In [13]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(900,950)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---
Problem 1/50: FAILED. Total solved: 0/50
Problem 2/50: SOLVED. Total solved: 1/50
Problem 3/50: FAILED. Total solved: 1/50
Problem 4/50: SOLVED. Total solved: 2/50
Problem 5/50: FAILED. Total solved: 2/50


<string>:3: SyntaxWarning: invalid escape sequence '\d'


Problem 6/50: FAILED. Total solved: 2/50
Problem 7/50: SOLVED. Total solved: 3/50
Problem 8/50: SOLVED. Total solved: 4/50
Problem 9/50: FAILED. Total solved: 4/50
Problem 10/50: FAILED. Total solved: 4/50
Problem 11/50: FAILED. Total solved: 4/50
Problem 12/50: FAILED. Total solved: 4/50
Problem 13/50: SOLVED. Total solved: 5/50
Problem 14/50: SOLVED. Total solved: 6/50
Problem 15/50: FAILED. Total solved: 6/50
Problem 16/50: SOLVED. Total solved: 7/50
Problem 17/50: FAILED. Total solved: 7/50
Problem 18/50: FAILED. Total solved: 7/50
Problem 19/50: SOLVED. Total solved: 8/50
Problem 20/50: FAILED. Total solved: 8/50
Problem 21/50: SOLVED. Total solved: 9/50
Problem 22/50: FAILED. Total solved: 9/50
Problem 23/50: FAILED. Total solved: 9/50
Problem 24/50: SOLVED. Total solved: 10/50
Problem 25/50: SOLVED. Total solved: 11/50
Problem 26/50: FAILED. Total solved: 11/50
Problem 27/50: FAILED. Total solved: 11/50
Problem 28/50: SOLVED. Total solved: 12/50
Problem 29/50: SOLVED. Total solv

In [11]:
k=1
pass_at_k = evaluate_pass_at_k(test_ds.select(range(950,974)).to_list(), k_value=k)
print(f"Pass@{k}: {pass_at_k}")


--- Evaluating Pass@1 ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Problem 1/24: FAILED. Total solved: 0/24
Problem 2/24: FAILED. Total solved: 0/24
Problem 3/24: FAILED. Total solved: 0/24
Problem 4/24: SOLVED. Total solved: 1/24
Problem 5/24: SOLVED. Total solved: 2/24
Problem 6/24: FAILED. Total solved: 2/24
Problem 7/24: FAILED. Total solved: 2/24
Problem 8/24: SOLVED. Total solved: 3/24
Problem 9/24: FAILED. Total solved: 3/24
Problem 10/24: SOLVED. Total solved: 4/24
Problem 11/24: SOLVED. Total solved: 5/24
Problem 12/24: SOLVED. Total solved: 6/24
Problem 13/24: FAILED. Total solved: 6/24
Problem 14/24: SOLVED. Total solved: 7/24
Problem 15/24: FAILED. Total solved: 7/24
Problem 16/24: SOLVED. Total solved: 8/24
Problem 17/24: FAILED. Total solved: 8/24
Problem 18/24: FAILED. Total solved: 8/24
Problem 19/24: SOLVED. Total solved: 9/24
Problem 20/24: SOLVED. Total solved: 10/24
Problem 21/24: FAILED. Total solved: 10/24
Problem 22/24: SOLVED. Total solved: 11/24
Problem 23/24: SOLVED. Total solved: 12/24
Problem 24/24: FAILED. Total solved: 12

Pass@10


In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(0,50)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



--- Evaluating Pass@10 ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 1/50: FAILED. Total solved: 0/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


...

Problem 47/50: FAILED. Total solved: 34/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 48/50: FAILED. Total solved: 34/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 49/50: SOLVED. Total solved: 35/50
Problem 50/50: SOLVED. Total solved: 36/50

Pass@10 = 0.7200
Pass@10: 0.72


In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(50,100)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



--- Evaluating Pass@10 ---


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 1/50: SOLVED. Total solved: 1/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


...

Problem 47/50: SOLVED. Total solved: 32/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 48/50: SOLVED. Total solved: 33/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 49/50: SOLVED. Total solved: 34/50
Problem 50/50: SOLVED. Total solved: 35/50

Pass@10 = 0.7000
Pass@10: 0.7


In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(100,150)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



--- Evaluating Pass@10 ---


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 1/50: SOLVED. Total solved: 1/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


...

Problem 19/50: FAILED. Total solved: 13/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 20/50: SOLVED. Total solved: 14/50


[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Problem 21/50: SOLVED. Total solved: 15/50


In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(150,200)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(200,250)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(250,300)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(300,350)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(350,400)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(400,450)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(450,500)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(500,550)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(550,600)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(600,650)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(650,700)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(700,750)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(750,800)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(800,850)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(850,900)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(900,950)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")

In [ ]:
k=10
pass_at_k = evaluate_pass_at_k(test_ds.select(range(950,1000)).to_list(), k_value=k)

print(f"Pass@{k}: {pass_at_k}")